# PS- S6E5

- Data: https://www.kaggle.com/competitions/playground-series-s6e3/data

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    OneHotEncoder
)

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)



from sklearn import set_config
set_config(display="diagram")

In [ ]:
# path
# train_path = r"/kaggle/input/competitions/playground-series-s6e3/train.csv
# test_path = r"/kaggle/input/competitions/playground-series-s6e3/test.csv"
# sub_path = r"/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv"


# path
train_path = r"C:\Users\Rudra\Desktop\kaggle\2026-mar\p6-s6e3\data\train.csv"
test_path = r"C:\Users\Rudra\Desktop\kaggle\2026-mar\p6-s6e3\data\test.csv"
sub_path = r"C:\Users\Rudra\Desktop\kaggle\2026-mar\p6-s6e3\data\sample_submission.csv"

# load data
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

display(train.sample(3), test.sample(3), sub.sample(3))

print(train.info())

display(train.describe(include="all").T)

train['Churn'].value_counts(normalize=True).plot.pie(autopct="%2.2f%%")
plt.show()


# Columns selection
object_cols = train.select_dtypes(include="object").columns.to_list()
object_cols.remove('Churn')

num_cols = train.select_dtypes(exclude="object").columns.to_list()
num_cols.remove('id')

target_col = 'Churn'

X = train.drop(columns=[target_col])
y = train[target_col]

y = y.map({'No':0, 'Yes':1})

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), object_cols),
        ("num", StandardScaler(), num_cols),
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

X_train, X_val,  y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2,
    stratify=y,
    random_state=42
)

display(model.fit(X_train, y_train))


# Prediction 
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:,1]

print(classification_report(y_val, y_pred))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred)
plt.show()

# Submission
test_preds = model.predict_proba(test)[:, 1]
sub['Churn'] = test_preds
sub.to_csv("submission.csv", index=False)


test_preds = model.predict_proba(test)[:, 1]

sub['Churn'] = test_preds
sub.to_csv("submission.csv", index=False)
print("submission.csv file saved!")